# 06 - Embedding Model Result Comparison

This notebook compares the downstream ML results for every embedding dataset produced in the previous notebooks. Some embedding datasets use `log_systemic_risk_label`, while the no-log datasets use `systemic_risk_label`. To keep the comparison fair, every trained model is scored in both spaces:

- `metric_space == "log"`: errors are measured after converting targets/predictions to log scale.
- `metric_space == "out"`: errors are measured on the original systemic-risk scale.

`native_space` records the target space used during model training.

In [ ]:
from pathlib import Path
import os
import sys

sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

PROJECT_ROOT = Path().resolve().parents[1]
print(f'Project root: {PROJECT_ROOT}')

## Dataset Registry

The `native_space` column says whether the dataset's training target is already log-transformed or in the original output scale.

In [ ]:
DATASETS = [
    {
        'dataset': 'GraphSAGE-v1-log',
        'embedding_model': 'GraphSAGE',
        'variant': 'v1-log',
        'filename': 'graphsage_srisk_dataset.parquet',
        'target_col': 'log_systemic_risk_label',
        'native_space': 'log',
    },
    {
        'dataset': 'Node2Vec-v1-log',
        'embedding_model': 'Node2Vec',
        'variant': 'v1-log',
        'filename': 'node2vec_srisk_dataset.parquet',
        'target_col': 'log_systemic_risk_label',
        'native_space': 'log',
    },
    {
        'dataset': 'GraphSAGE-v2-log',
        'embedding_model': 'GraphSAGE',
        'variant': 'v2-log',
        'filename': 'graphsage_fixed_srisk_dataset.parquet',
        'target_col': 'log_systemic_risk_label',
        'native_space': 'log',
    },
    {
        'dataset': 'GraphSAGE-v3-log',
        'embedding_model': 'GraphSAGE',
        'variant': 'v3-log',
        'filename': 'graphsage_fixed_128_srisk_dataset.parquet',
        'target_col': 'log_systemic_risk_label',
        'native_space': 'log',
    },
    {
        'dataset': 'Node2Vec-v2-log',
        'embedding_model': 'Node2Vec',
        'variant': 'v2-log',
        'filename': 'node2vec_fixed_srisk_dataset.parquet',
        'target_col': 'log_systemic_risk_label',
        'native_space': 'log',
    },
    {
        'dataset': 'Node2Vec-v3-log',
        'embedding_model': 'Node2Vec',
        'variant': 'v3-log',
        'filename': 'node2vec_fixed_v3_srisk_dataset.parquet',
        'target_col': 'log_systemic_risk_label',
        'native_space': 'log',
    },
    {
        'dataset': 'GraphSAGE-v2-nolog',
        'embedding_model': 'GraphSAGE',
        'variant': 'v2-nolog',
        'filename': 'graphsage_fixed_32_srisk_nolog_dataset.parquet',
        'target_col': 'systemic_risk_label',
        'native_space': 'out',
    },
    {
        'dataset': 'GraphSAGE-v3-nolog',
        'embedding_model': 'GraphSAGE',
        'variant': 'v3-nolog',
        'filename': 'graphsage_fixed_64_srisk_nolog_dataset.parquet',
        'target_col': 'systemic_risk_label',
        'native_space': 'out',
    },
    {
        'dataset': 'GraphSAGE-v4-nolog',
        'embedding_model': 'GraphSAGE',
        'variant': 'v4-nolog',
        'filename': 'graphsage_fixed_128_srisk_nolog_dataset.parquet',
        'target_col': 'systemic_risk_label',
        'native_space': 'out',
    },
    {
        'dataset': 'Node2Vec-v2-nolog',
        'embedding_model': 'Node2Vec',
        'variant': 'v2-nolog',
        'filename': 'node2vec_fixed_32_srisk_nolog_dataset.parquet',
        'target_col': 'systemic_risk_label',
        'native_space': 'out',
    },
    {
        'dataset': 'Node2Vec-v3-nolog',
        'embedding_model': 'Node2Vec',
        'variant': 'v3-nolog',
        'filename': 'node2vec_fixed_64_srisk_nolog_dataset.parquet',
        'target_col': 'systemic_risk_label',
        'native_space': 'out',
    },
    {
        'dataset': 'Node2Vec-v4-nolog',
        'embedding_model': 'Node2Vec',
        'variant': 'v4-nolog',
        'filename': 'node2vec_fixed_128_srisk_nolog_dataset.parquet',
        'target_col': 'systemic_risk_label',
        'native_space': 'out',
    },
]

pd.DataFrame(DATASETS)

## Load Embedding Datasets

In [3]:
loaded = {}
dataset_summary = []

for spec in DATASETS:
    df, feature_cols = load_gnn_dataset(
        PROJECT_ROOT,
        target_col=spec['target_col'],
        filename=spec['filename'],
    )
    loaded[spec['dataset']] = {'spec': spec, 'df': df, 'feature_cols': feature_cols}
    dataset_summary.append({
        **spec,
        'rows': len(df),
        'embedding_cols': len(feature_cols),
        'period_min': df['period'].min(),
        'period_max': df['period'].max(),
    })

dataset_summary = pd.DataFrame(dataset_summary)
dataset_summary.sort_values(['native_space', 'embedding_model', 'variant', 'dataset']).reset_index(drop=True)

,dataset,embedding_model,variant,filename,target_col,native_space,rows,embedding_cols,period_min,period_max
0,graphsage_fixed_32_log,GraphSAGE,fixed_32,graphsage_fixed_srisk_dataset.parquet,log_systemic_risk_label,log,145536,32,2016Q1,2023Q4
1,graphsage_original,GraphSAGE,original,graphsage_srisk_dataset.parquet,log_systemic_risk_label,log,145536,64,2016Q1,2023Q4
2,node2vec_fixed_128_log,Node2Vec,fixed_128,node2vec_fixed_v3_srisk_dataset.parquet,log_systemic_risk_label,log,145536,128,2016Q1,2023Q4
3,node2vec_fixed_32_log,Node2Vec,fixed_32,node2vec_fixed_srisk_dataset.parquet,log_systemic_risk_label,log,145536,32,2016Q1,2023Q4
4,node2vec_original,Node2Vec,original,node2vec_srisk_dataset.parquet,log_systemic_risk_label,log,145536,64,2016Q1,2023Q4
5,graphsage_fixed_32_out,GraphSAGE,fixed_32,graphsage_fixed_32_srisk_nolog_dataset.parquet,systemic_risk_label,out,145536,32,2016Q1,2023Q4
6,graphsage_fixed_64_out,GraphSAGE,fixed_64,graphsage_fixed_64_srisk_nolog_dataset.parquet,systemic_risk_label,out,145536,64,2016Q1,2023Q4
7,node2vec_fixed_32_out,Node2Vec,fixed_32,node2vec_fixed_32_srisk_nolog_dataset.parquet,systemic_risk_label,out,145536,32,2016Q1,2023Q4
8,node2vec_fixed_64_out,Node2Vec,fixed_64,node2vec_fixed_64_srisk_nolog_dataset.parquet,systemic_risk_label,out,145536,64,2016Q1,2023Q4


## Candidate Models

In [ ]:
candidate_models = {
    'Linear Regression': make_pipeline(LinearRegression()),
    'Ridge':             make_pipeline(Ridge(alpha=1.0)),
    'MLP':               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    'Random Forest':     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    'Gradient Boosting': make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    'XGBoost':           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

## Scoring Helpers

For `out -> log` conversion, negative original-scale predictions are clipped at zero only for the log-space metric conversion. Original-scale metrics still use the raw model prediction.

In [5]:
def regression_scores(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
        'r2': r2_score(y_true, y_pred),
    }


def values_in_metric_space(y_true_native, y_pred_native, native_space, metric_space):
    y_true_native = np.asarray(y_true_native, dtype=float)
    y_pred_native = np.asarray(y_pred_native, dtype=float)

    if native_space == metric_space:
        return y_true_native, y_pred_native

    if native_space == 'log' and metric_space == 'out':
        return np.expm1(y_true_native), np.expm1(y_pred_native)

    if native_space == 'out' and metric_space == 'log':
        return np.log1p(np.clip(y_true_native, 0, None)), np.log1p(np.clip(y_pred_native, 0, None))

    raise ValueError(f'Unsupported conversion: {native_space} -> {metric_space}')


def score_model_split(model, split_df, feature_cols, target_col, native_space, metric_space):
    y_true_native = split_df[target_col]
    y_pred_native = model.predict(split_df[feature_cols])
    y_true, y_pred = values_in_metric_space(y_true_native, y_pred_native, native_space, metric_space)
    return regression_scores(y_true, y_pred)

## Train and Compare All Embedding Models

In [ ]:
records = []
trainers = {}

for dataset_name, item in loaded.items():
    spec = item['spec']
    df = item['df']
    feature_cols = item['feature_cols']

    trainer = ModelTrainer(df=df, feature_cols=feature_cols, target_col=spec['target_col'])
    trainer.train_all(candidate_models)
    trainers[dataset_name] = trainer

    for model_name, model in trainer.models.items():
        for metric_space in ['log', 'out']:
            row = {
                'dataset': dataset_name,
                'embedding_model': spec['embedding_model'],
                'variant': spec['variant'],
                'embedding_cols': len(feature_cols),
                'native_space': spec['native_space'],
                'metric_space': metric_space,
                'model': model_name,
            }

            for split_name, split_df in [
                ('train', trainer.train_df),
                ('validation', trainer.val_df),
            ]:
                scores = score_model_split(
                    model=model,
                    split_df=split_df,
                    feature_cols=feature_cols,
                    target_col=spec['target_col'],
                    native_space=spec['native_space'],
                    metric_space=metric_space,
                )
                for metric_name, metric_value in scores.items():
                    row[f'{split_name}_{metric_name}'] = metric_value

            records.append(row)

all_results = pd.DataFrame(records)
DISPLAY_COLS = [
    'dataset', 'embedding_model', 'variant', 'embedding_cols',
    'native_space', 'metric_space', 'model',
    'train_mae', 'validation_mae',
    'train_rmse', 'validation_rmse',
    'train_r2', 'validation_r2',
]

all_results[DISPLAY_COLS].sort_values(['metric_space', 'validation_rmse', 'validation_mae']).reset_index(drop=True)

## Log-Space Leaderboard

In [7]:
log_results = all_results.query("metric_space == 'log'").copy()
log_results[DISPLAY_COLS].sort_values(['validation_rmse', 'validation_mae']).reset_index(drop=True)

,dataset,embedding_model,variant,embedding_cols,native_space,metric_space,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,node2vec_fixed_128_log,Node2Vec,fixed_128,128,log,log,random_forest,0.006494,0.030423,0.025817,0.031862,0.129201,0.102832,0.967452,0.700868,0.738357
1,node2vec_fixed_128_log,Node2Vec,fixed_128,128,log,log,xgboost,0.011188,0.029773,0.024470,0.050109,0.130405,0.101163,0.919493,0.695269,0.746784
2,node2vec_fixed_64_out,Node2Vec,fixed_64,64,out,log,random_forest,0.007537,0.031478,0.029549,0.035766,0.130863,0.114768,0.958986,0.693121,0.674092
3,node2vec_fixed_32_log,Node2Vec,fixed_32,32,log,log,random_forest,0.006772,0.032153,0.030127,0.032031,0.131996,0.114281,0.967105,0.687785,0.676853
4,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,log,random_forest,0.007942,0.035185,0.035266,0.037012,0.136176,0.125540,0.956079,0.667699,0.610045
5,node2vec_fixed_32_log,Node2Vec,fixed_32,32,log,log,xgboost,0.013355,0.030482,0.026345,0.062539,0.136432,0.112552,0.874600,0.666447,0.686555
6,node2vec_fixed_64_out,Node2Vec,fixed_64,64,out,log,xgboost,0.015296,0.034692,0.031551,0.061114,0.138507,0.124926,0.880248,0.656226,0.613851
7,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,log,xgboost,0.016089,0.037414,0.035908,0.066326,0.147236,0.133681,0.858955,0.611531,0.557828
8,node2vec_original,Node2Vec,original,64,log,log,random_forest,0.008466,0.038476,0.030545,0.040040,0.184099,0.147009,0.948598,0.392657,0.465261
9,node2vec_fixed_32_log,Node2Vec,fixed_32,32,log,log,linear_regression,0.058798,0.080441,0.074960,0.144037,0.197041,0.167245,0.334815,0.304266,0.307922


## Original-Scale (`out`) Leaderboard

In [8]:
out_results = all_results.query("metric_space == 'out'").copy()
out_results[DISPLAY_COLS].sort_values(['validation_rmse', 'validation_mae']).reset_index(drop=True)

,dataset,embedding_model,variant,embedding_cols,native_space,metric_space,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,out,random_forest,0.031658,0.156183,0.147287,0.341280,1.114571,1.129883,0.966533,0.866449,0.674042
1,node2vec_fixed_64_out,Node2Vec,fixed_64,64,out,out,random_forest,0.029302,0.149306,0.119254,0.322992,1.160822,0.848385,0.970024,0.855135,0.816227
2,node2vec_fixed_128_log,Node2Vec,fixed_128,128,log,out,random_forest,0.030585,0.153138,0.132928,0.479702,1.244759,1.385212,0.933880,0.833428,0.510078
3,node2vec_fixed_128_log,Node2Vec,fixed_128,128,log,out,xgboost,0.036302,0.154222,0.107992,0.239300,1.401684,0.975655,0.983546,0.788782,0.756955
4,node2vec_fixed_64_out,Node2Vec,fixed_64,64,out,out,xgboost,0.043733,0.170745,0.125627,0.219575,1.471841,0.873980,0.986147,0.767109,0.804972
5,node2vec_fixed_32_log,Node2Vec,fixed_32,32,log,out,random_forest,0.030307,0.172021,0.135181,0.424323,1.580922,1.190057,0.948265,0.731310,0.638399
6,node2vec_fixed_32_log,Node2Vec,fixed_32,32,log,out,xgboost,0.043950,0.179892,0.124987,0.318644,1.921386,1.145960,0.970825,0.603119,0.664700
7,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,out,xgboost,0.047181,0.202500,0.150238,0.253275,1.974640,1.096034,0.981568,0.580814,0.693280
8,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,out,linear_regression,0.383542,0.540306,0.450386,1.471555,2.437767,1.461340,0.377780,0.361125,0.454749
9,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,out,ridge,0.383487,0.540223,0.450285,1.471555,2.437915,1.461422,0.377780,0.361048,0.454688


## Best Model Per Embedding Dataset

In [9]:
best_by_dataset = (
    all_results
    .sort_values(['metric_space', 'dataset', 'validation_rmse', 'validation_mae'])
    .groupby(['metric_space', 'dataset'], as_index=False)
    .first()
)

best_by_dataset[DISPLAY_COLS].sort_values(['metric_space', 'validation_rmse', 'validation_mae']).reset_index(drop=True)

,dataset,embedding_model,variant,embedding_cols,native_space,metric_space,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,node2vec_fixed_128_log,Node2Vec,fixed_128,128,log,log,random_forest,0.006494,0.030423,0.025817,0.031862,0.129201,0.102832,0.967452,0.700868,0.738357
1,node2vec_fixed_64_out,Node2Vec,fixed_64,64,out,log,random_forest,0.007537,0.031478,0.029549,0.035766,0.130863,0.114768,0.958986,0.693121,0.674092
2,node2vec_fixed_32_log,Node2Vec,fixed_32,32,log,log,random_forest,0.006772,0.032153,0.030127,0.032031,0.131996,0.114281,0.967105,0.687785,0.676853
3,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,log,random_forest,0.007942,0.035185,0.035266,0.037012,0.136176,0.125540,0.956079,0.667699,0.610045
4,node2vec_original,Node2Vec,original,64,log,log,random_forest,0.008466,0.038476,0.030545,0.040040,0.184099,0.147009,0.948598,0.392657,0.465261
5,graphsage_original,GraphSAGE,original,64,log,log,random_forest,0.006973,0.034803,0.028440,0.038312,0.206160,0.172369,0.952940,0.238379,0.264864
6,graphsage_fixed_32_log,GraphSAGE,fixed_32,32,log,log,random_forest,0.006006,0.129821,0.121977,0.033576,0.354567,0.298949,0.963855,-1.252823,-1.211295
7,graphsage_fixed_32_out,GraphSAGE,fixed_32,32,out,log,xgboost,0.017944,0.325346,0.479794,0.068363,0.786384,0.968642,0.850157,-10.081499,-22.215495
8,graphsage_fixed_64_out,GraphSAGE,fixed_64,64,out,log,linear_regression,0.155504,0.446183,0.558040,0.256000,0.855312,1.021719,-1.101243,-12.109288,-24.829404
9,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,out,random_forest,0.031658,0.156183,0.147287,0.341280,1.114571,1.129883,0.966533,0.866449,0.674042


## Compact Pivot View

This table keeps the best model for each dataset and shows validation RMSE and MAE side by side under `log` and `out` metric spaces.

In [ ]:
pivot = (
    best_by_dataset
    .pivot_table(
        index=['dataset', 'embedding_model', 'variant', 'embedding_cols', 'native_space'],
        columns='metric_space',
        values=['validation_rmse', 'validation_mae'],
        aggfunc='first',
    )
)

pivot.columns = [f'{metric}_{space}' for metric, space in pivot.columns]
pivot = pivot.reset_index()

pivot.sort_values(['validation_rmse_log', 'validation_mae_log']).reset_index(drop=True)

## Overall Winners

In [11]:
overall_winners = (
    all_results
    .sort_values(['metric_space', 'validation_rmse', 'validation_mae'])
    .groupby('metric_space', as_index=False)
    .first()
)

overall_winners[DISPLAY_COLS]

,dataset,embedding_model,variant,embedding_cols,native_space,metric_space,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,node2vec_fixed_128_log,Node2Vec,fixed_128,128,log,log,random_forest,0.006494,0.030423,0.025817,0.031862,0.129201,0.102832,0.967452,0.700868,0.738357
1,node2vec_fixed_32_out,Node2Vec,fixed_32,32,out,out,random_forest,0.031658,0.156183,0.147287,0.341280,1.114571,1.129883,0.966533,0.866449,0.674042
